## Visualize the cells of one spheroid in 3D. 


In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.ticker as ticker
%matplotlib inline
# %matplotlib widget

# Set current working directory


In [ ]:
# Parameters. run_all.py sweeps cell_line via the environment so one notebook emits
# both Fig 2f (HCT116) and Suppl 1d (HT29); the default keeps interactive use
# unchanged. The example well and barcode follow from the cell line, so they are a
# lookup rather than three variables to keep in sync — upstream the HCT116 block was
# commented out, which is why only one of the two panels was ever produced.
import os

EXAMPLE_WELL = {
    'HCT116': ('D12', 'PB000137'),   # TODO (from upstream): double-check the example images
    'HT29':   ('K11', 'PB000142'),
}

cell_line = os.environ.get('COLOPAINT3D_CELL_LINE', 'HT29')   # 'HCT116' or 'HT29'
well, barcode = EXAMPLE_WELL[cell_line]
print(f'cell_line={cell_line}  well={well}  barcode={barcode}')

In [ ]:
file = features("exp1_main", "011225", "SingleCell", f"{cell_line}.parquet")
df = pd.read_parquet(file) # Go for one spheroid at first

In [ ]:
# Save the data
ImagesOut = str(figdir('Fig2')) + '/'
# HCT116 is the main-figure panel; HT29 is the supplementary counterpart.
SPHEROID_PANEL = {'HCT116': 'Fig2f', 'HT29': 'SupplFig1d'}

if not os.path.exists(ImagesOut): 
        os.makedirs(ImagesOut)

In [ ]:
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
dpi = 300
figformat = 'pdf'

### Do the plotting

In [ ]:
dfSingleSpheroid = df.query('Metadata_Well == @well & Metadata_Barcode == @barcode')[['Metadata_Site', 'Location_Center_X_cells', 'Location_Center_Y_cells']]

In [ ]:
sns.set(style = "white")

fig = plt.figure(figsize=(10,10))
ax = plt.axes(projection='3d')

x = dfSingleSpheroid['Location_Center_X_cells'] # X_location
y = dfSingleSpheroid['Location_Center_Y_cells'] # Y_location
z = dfSingleSpheroid['Metadata_Site'] # plane

x_res = 0.227 # µm per pixel
y_res = 0.227 # µm per pixel
z_res = 5 # µm per plane

ax.set_xlabel("µm")
ax.set_ylabel("µm") 
ax.set_zlabel("µm") 

# cm = plt.cm.get_cmap('YlOrRd')
scatter_plot = ax.scatter3D(x, y, z, c=dfSingleSpheroid['Metadata_Site'],  cmap='YlOrRd', s=20, alpha=0.8)
# plt.colorbar(scatter_plot, label='Slice')


zticks = ticker.FuncFormatter(lambda x, pos: '{0:g}'.format(x*z_res))
ax.zaxis.set_major_formatter(zticks)

ax.invert_zaxis()

xticks = ticker.FuncFormatter(lambda x, pos: '{0:.1f}'.format(x*x_res))
ax.xaxis.set_major_formatter(xticks)

yticks = ticker.FuncFormatter(lambda x, pos: '{0:.1f}'.format(x*y_res))
ax.yaxis.set_major_formatter(yticks)

m = max(y*x_res)

plt.xticks(np.arange(0, m, 30)/x_res)
plt.yticks(np.arange(0, m, 30)/y_res)
ax.set_zticks(np.arange(0, max(z*z_res), 15)/z_res)
ax.set_box_aspect((np.ptp(x*x_res), np.ptp(y*x_res), np.ptp(z*z_res)))



# Set initial view angle
ax.view_init(elev=15, azim=45)  

save_panel(fig, SPHEROID_PANEL[cell_line], data=dfSingleSpheroid,
           caption=f'Detected cell centroids through a spheroid, {cell_line}',
           notebook='analysis/3_Figure2/CellDetectionSanityCheck/3_Plot_Spheroids.ipynb')

plt.show()